### **Customer Data**

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import *

In [0]:
df=spark.read.format("delta").option("path","/Volumes/workspace/bronze_olist/customers/data/").load()

In [0]:
df.printSchema()

In [0]:
display(df.select([sum(col(i).isNull().cast("int")).alias(i) for i in df.columns]))

In [0]:
df_silver=df.drop(col("_rescued_data"))\
    .withColumn("customer_city",trim(initcap(lower("customer_city"))))\
    .withColumn("customer_state",trim(("customer_state")))\
    .withColumn("customer_zip_code_prefix",trim(col("customer_zip_code_prefix")))\
    .withColumn("modified_date",current_timestamp())
    

    
    
  

### **Product Data**

In [0]:
df_product=spark.read.format("delta").option("path","/Volumes/workspace/bronze_olist/products/data/").option("header",True).option("inferSchema",True).load()

In [0]:
df_product.printSchema()

In [0]:
display(df_product)

In [0]:
df_product.select([(sum(col(i).isNull().cast("int")).alias(i))for i in df_product.columns]).show()

### **Category Data**

In [0]:
df_cat=spark.read.format("delta").option("path","/Volumes/workspace/bronze_olist/categories/data/").load()

In [0]:
df_cat.printSchema()

In [0]:
df_product_enriched=df_product.join(df_cat,on="product_category_name",how="left")

In [0]:
df_product_enriched\
    .drop("_rescued_data")\
    .withColumn("product_category_name_english",trim(initcap(lower("product_category_name_english"))))\
    .withColumn("product_category_name",trim(initcap(lower("product_category_name"))))\
    .withColumn("product_category_name",when(col("product_category_name").isNull(),"UNKNOWN").otherwise(col("product_category_name")))\
    .withColumn("product_category_name_english",when(col("product_category_name_english").isNull(),"UNKNOWN").otherwise(col("product_category_name_english")))\
    .withColumnRenamed("product_description_lenght","product_description_length")\
    .withColumnRenamed("product_name_lenght","product_name_length")\
    .withColumn("product_name_length",col("product_name_length").cast("int"))\
    .withColumn("product_description_length",col("product_description_length").cast("int"))\
    .withColumn("product_photos_qty",col("product_photos_qty").cast("int"))\
    .withColumn("product_weight_g",col("product_weight_g").cast("int"))\
    .withColumn("product_length_cm",col("product_length_cm").cast("int"))\
    .withColumn("product_height_cm",col("product_height_cm").cast("int"))\
    .withColumn("product_width_cm",col("product_width_cm").cast("int"))\
    .withColumn("modified_date",current_timestamp())
    


### **Order Items Data**

In [0]:
df_order_items=spark.read.format("delta").option("path","/Volumes/workspace/bronze_olist/order_items/data").load()

In [0]:
df_order_items.printSchema()

In [0]:
display(df_order_items)

In [0]:
df_order_items.select([((sum(col(i).isNull().cast("int")).alias(i)))for i in df_order_items.columns]).show()

In [0]:
df_order_items.groupBy(col("order_id"),col("order_item_id")).count().filter("count>1").show()

In [0]:
df_order_items.withColumn("freight_value",col("freight_value").cast("double")).filter("freight_value<0").show()

In [0]:
df_order_items.drop("_rescued_data")\
    .withColumn("order_id",trim(col("order_id")))\
    .withColumn("order_item_id",col("order_item_id").cast(int))\
    .withColumn("product_id",trim(col("product_id")))\
    .withColumn("seller_id",trim(col("seller_id")))\
    .withColumn("shipping_limit_date",col("shipping_limit_date").cast("timestamp"))\
    .withColumn("price",col("price").cast("double"))\
    .withColumn("freight_value",col("freight_value").cast("double"))\
    .withColumn("modified_date",current_timestamp())